Checking out strategies to minimise the loss function in training.  

The main problem we are facing right now is the model getting stuck in a local minima. I've to see what all i can do.  

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import random
import torch
from pt_to_api.mnist import SimpleMNIST
from pt_to_api.utils import show_single_channel_red_green_black as S, to_show_list as tsl
import gc
import pickle

In [ ]:
MODEL_PATH = Path("../../pt-to-api/data/model.pt")
INPUT_PATH = Path("../../pt-to-api/data/first-input-tens.pt")
MAIN_OUT_DIR = (Path.cwd() / "mnist-patches-data")
MAIN_OUT_DIR.mkdir(parents=True, exist_ok=True)

inp = torch.load(INPUT_PATH, weights_only=False)
model = SimpleMNIST()
model.load_state_dict(torch.load(MODEL_PATH))

In [ ]:
PATCHES_DATA_ROOT = Path("/Users/hariomnarang/Desktop/personal/hiccup-ide/backend/notebooks/mnist-patches-data")
layer_name = "layers.2"
channel = 12
SHAPE = (8,9)
MAIN_DUMP_DIR = Path("runs-l2-o12")
MAIN_DUMP_DIR.mkdir(parents=True, exist_ok=True)


patches_ds = PATCHES_DATA_ROOT / layer_name / str(channel)
assert patches_ds.is_dir()

In [ ]:
def _mse(x, codes, comps):
    diff = x - (codes@comps)
    return (diff**2).mean()


def get_comp_scores(X, codes, components):
    main_mse = _mse(X, codes, components)
    scores = []
    for i in range(len(components)):
        comps = components.copy()
        comps[i].fill(0)
        new_mse = _mse(X, codes, comps)
        comp_score = new_mse - main_mse
        scores.append(comp_score)
    return scores, main_mse


def get_comp_scores_when_active(X, codes, components):
    scores = []
    for i in range(len(components)):
        idxs = get_indices_where_comp_is_active(i, codes, components)
        _X, _codes = X[idxs], codes[idxs]

        main_mse = _mse(_X, _codes, components)
        comps = components.copy()
        comps[i].fill(0)
        new_mse = _mse(_X, _codes, comps)
        # ratio implicitly
        comp_score = (new_mse - main_mse) / new_mse

        scores.append(comp_score)
    return scores

def get_indices_where_comp_is_active(comp_idx, codes, components):
    comp_idx = 3

    comp = components[comp_idx]
    from pt_to_api.utils import otsu_threshold
    thresh = otsu_threshold(codes[:, comp_idx])

    indices = np.argwhere(codes[:, comp_idx] > thresh)
    return indices.reshape(-1)

In [ ]:
def get_sampled_patches(num_samples=3000):
    patches = []
    for p in patches_ds.glob("*.pt"):
        patches.append(torch.load(p, weights_only=False, map_location="cpu").numpy())
    patches = np.concat(patches)
    samples_idxs = torch.randperm(patches.shape[0]).numpy()
    samples = patches[samples_idxs[:num_samples]]
    return samples

# # only run once, uncomment if you wanna generate new samples
# patches = get_sampled_patches(7000)
# torch.save(patches, "./saved-patches-7000.pt")

In [ ]:
from pt_to_api.benchmark import MeanPerDimGlobalStdScaler, NormaliseStdScaler
from sklearn.decomposition import FastICA,PCA
from sklearn.preprocessing import StandardScaler
from pt_to_api.benchmark import train_x as TX
from dataclasses import dataclass
from pt_to_api import benchmark as B

layer = model.get_submodule(f"{layer_name}")
weight = layer.weight[channel].clone().detach()
weight = weight.reshape(-1).numpy()


patches = torch.load(MAIN_DUMP_DIR / "saved-patches-7000.pt", weights_only=False)
pw = patches * weight
scaler = NormaliseStdScaler().fit(pw)
scaled_pw = scaler.transform(pw)

In [ ]:
pca = PCA(n_components=50, whiten=False)
pca.fit(scaled_pw)
with np.printoptions(suppress=True, precision=5):
    print(pca.explained_variance_ratio_)
S([c.reshape(8,9) for c in pca.components_[:8]], 20, 8)
plt.show()

## Warmup

Doesnt solve the problem of having random duplicate stuff.  

In [ ]:
import pickle

dump_dir = MAIN_DUMP_DIR / "warmup"

c2r = {}

for i in range(2, 16):
    d = dump_dir /f"r{i}.pkl"
    with open(d, "rb") as f:
        run = pickle.load(f)
        c2r[i] = run

In [ ]:
plt.plot(list(range(2,16)), [c2r[i].loss for i in range(2,16)])
plt.show()

In [ ]:
import math

for i in range(2,16):
    cols = min(i, 10)
    rows = math.ceil(i / cols)
    rsize = 2*rows
    csize = 20
    scores = np.array(get_comp_scores(scaled_pw, c2r[i].codes, c2r[i].components))
    scores = (100*scores) / scores.sum()

    print("shape", i)
    S([c.reshape(SHAPE) for c in c2r[i].components], (csize, rsize), cols, viztype="local", suptitle=f"loss: {c2r[i].loss}", ax_titles=[f"{s:.3f}" for s in scores])
    plt.show()

# aneal on unnormalised data

In [ ]:
from pt_to_api.benchmark import NormaliseStdScaler
from sklearn.decomposition import FastICA,PCA
from sklearn.preprocessing import StandardScaler
from pt_to_api.benchmark import train_x as TX
from dataclasses import dataclass
from pt_to_api import benchmark as B

# layer = model.get_submodule(f"{layer_name}")
# weight = layer.weight[channel].clone().detach()
# weight = weight.reshape(-1).numpy()


MAIN_DUMP_DIR = Path("runs-l2-o12")
patches = torch.load(MAIN_DUMP_DIR / "saved-patches-7000.pt", weights_only=False)
pw = patches * weight

scaler = NormaliseStdScaler().fit(pw)
scaled_pw = scaler.transform(pw)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from scipy.optimize import linear_sum_assignment

# would like to see the stability score plot also actually
def show_closest_component_of_W_for_each_component(components, W_true, figsize=(5, 2)):
    """
    Given two arrays of numpy vectors of same shapes
    for every component in `components`, this function shows the array in `W_true`
    which has the maximum cosine similarity with the component
    """
    sims = np.abs(cosine_similarity(components, W_true))
    pairs = []
    for i in range(len(components)):
        j = np.argmax(sims[i])
        pairs.append((i, j, sims[i][j]))
    for i, j, score in pairs:
        S(
            [components[i].reshape(3, 3), W_true[j].reshape(3, 3)],
            figsize,
            mode=MODE,
            suptitle=f"similarity score={score}",
            ax_titles=["component", "ground_truth"],
            viztype="local",
        )
        plt.show()


def evaluate_recovery(W_learned, W_true, threshold=0.95):
    """
    W_learned: (n_atoms, patch_dim)
    W_true: (n_atoms, patch_dim)
    
    for each true atom, finds the best matching learned atom by cosine similarity
    returns fraction of true atoms recovered above threshold
    """
    W_l = W_learned / (np.linalg.norm(W_learned, axis=1, keepdims=True) + 1e-8)
    W_t = W_true / (np.linalg.norm(W_true, axis=1, keepdims=True) + 1e-8)
    
    sim = np.abs(W_l @ W_t.T)  # (n_atoms, n_atoms), abs because sign is arbitrary
    best_match = sim.max(axis=0)  # for each true atom, best cosine with any learned atom
    
    recovered = (best_match >= threshold).mean()
    print(f"Mean best cosine similarity: {best_match.mean():.4f}")
    print(f"Fraction recovered (>{threshold}): {recovered:.4f}")
    return best_match, recovered


def _match_atoms(D1: np.ndarray, D2: np.ndarray):
    """
    Match atoms of D1 to atoms of D2 using the Hungarian algorithm
    on cosine distances. Assumes square dictionaries (same n_components).

    D1, D2: shape (n_components, n_features) — sklearn's components_ layout.

    Returns:
        row_ind, col_ind: matched index arrays
        matched_similarities: per-pair cosine similarities
        mean_sim: mean cosine similarity across matched pairs
    """
    # Guard against dead atoms (zero-norm rows produce NaN cosine distances)
    norms_1 = np.linalg.norm(D1, axis=1, keepdims=True)
    norms_2 = np.linalg.norm(D2, axis=1, keepdims=True)
    if np.any(norms_1 == 0) or np.any(norms_2 == 0):
        raise ValueError(
            "One or more atoms have zero norm. "
            "Remove or replace dead atoms before matching."
        )

    # cost = cosine_distances(D1, D2)          # shape (n_components, n_components), values in [0, 2]
    cost = np.abs(cosine_similarity(D1, D2))
    cost = 1 - cost

    row_ind, col_ind = linear_sum_assignment(cost)
    matched_similarities = 1 - cost[row_ind, col_ind]
    mean_sim = float(matched_similarities.mean())
    return row_ind, col_ind, matched_similarities, mean_sim

def get_live(components):
    dead = find_dead_atoms(components).numpy()
    live = np.array([i for i in range(components.shape[0]) if i not in dead])
    return components[live]

def hungarian_match(all_components: list[np.ndarray]):
    """
    Pairwise similarity matching across runs using the Hungarian algorithm.

    Args:
        all_components: list of arrays, each shape (n_components, n_features).
                        All arrays must have the same shape.

    Returns:
        upper: 1-D array of pairwise similarities for all unique pairs
        stability_score: mean of upper
        best_run_idx: index of the run most similar to all others
        pairwise_sims: (n_runs, n_runs) symmetric similarity matrix, diagonal = 1
    """
    n_runs = len(all_components)

    if n_runs < 2:
        raise ValueError("Need at least 2 runs to compute pairwise similarity.")

    shapes = [d.shape for d in all_components]
    if len(set(shapes)) != 1:
        raise ValueError(
            f"All dictionaries must have the same shape. Got: {shapes}"
        )

    pairwise_sims = np.ones((n_runs, n_runs))
    for i in range(n_runs):
        for j in range(i + 1, n_runs):
            _, _, _, mean_sim = _match_atoms(all_components[i], all_components[j])
            pairwise_sims[i, j] = mean_sim
            pairwise_sims[j, i] = mean_sim

    upper = pairwise_sims[np.triu_indices(n_runs, k=1)]
    stability_score = float(upper.mean())

    # Exclude self-similarity (diagonal=1) when ranking runs
    np.fill_diagonal(pairwise_sims, 0)
    mean_sim_per_run = pairwise_sims.sum(axis=1) / (n_runs - 1)
    best_run_idx = int(np.argmax(mean_sim_per_run))
    np.fill_diagonal(pairwise_sims, 1)  # restore diagonal

    return upper, stability_score, best_run_idx, pairwise_sims

def find_dead_atoms(W, threshold=0.1):
    if not isinstance(W, torch.Tensor):
        W = torch.tensor(W)
    
    peak = W.abs().max(dim=1).values
    peak_normalised = peak / peak.max()
    
    return torch.where(peak_normalised < threshold)[0]

In [ ]:
import pickle
import gc

dump_dir = MAIN_DUMP_DIR / "cosine-anneal-normalize-std-test"

anneal_c2r = {}

for i in range(9, 16):
    for seed in range(3):
        d = dump_dir / "comps" / str(i) /f"seed_{seed}.pkl"
        with open(d, "rb") as f:
            run = pickle.load(f)
            anneal_c2r[(i, seed)] = run
gc.collect()

In [ ]:
# put the seed and comp
# prepare components
comp_idx_by_identity = {}
all_comps = []
for n in range(9,16):
    for seed in range(2):
        run = anneal_c2r[n,seed]
        for comp_idx, comp in enumerate(run.components):
            idx = len(all_comps)
            all_comps.append(comp)
            comp_idx_by_identity[idx] = (n, seed, comp_idx)

In [ ]:
from sklearn.preprocessing import normalize

X_reduced = normalize(all_comps, "l2")
abs_sim = np.abs(cosine_similarity(X_reduced, X_reduced))  # (n_samples, n_samples)
distance_matrix = 1 - abs_sim  # values in [0, 1]

In [ ]:
# from sklearn.cluster import DBSCAN

# clustering = DBSCAN(metric='precomputed', eps=0.3, min_samples=5)
# labels = clustering.fit_predict(distance_matrix)
from sklearn.cluster import HDBSCAN

hdbscan = HDBSCAN(copy=True, min_cluster_size=5, metric="precomputed")

hdbscan.fit(distance_matrix)

In [ ]:
comp2score_when_active = {}
for n in range(9,16):
    for seed in range(2):
        run = anneal_c2r[n,seed]
        scores = get_comp_scores_when_active(scaled_pw, run.codes, run.components)
        scores = np.array(scores)
        for j in range(len(scores)):
            comp2score_when_active[n, seed, j] = scores[j]

In [ ]:
comp2score = {}
for n in range(9,16):
    for seed in range(2):
        run = anneal_c2r[n,seed]
        scores, _ = get_comp_scores(scaled_pw, run.codes, run.components)
        scores = np.array(scores)
        scores = 100*scores / scores.sum()
        for j in range(len(scores)):
            comp2score[n, seed, j] = scores[j]

In [ ]:
# this is good, might wanna cleanup stuff now
# from each stable cluster, find the one with the maximum score
# use as a component
# and rerun, easy.   
# we can actually do it per seed also btw :)
# idk if that is useful
# i should try
from collections import defaultdict
import math

l2comps = defaultdict(list)
l2_maxscores = {}

for i, label in enumerate(hdbscan.labels_):
    n, seed, comp_idx = comp_idx_by_identity[i]
    l2comps[label].append(((n, seed, comp_idx), all_comps[i]))

for label, vals in l2comps.items():
    comps, titles = [], []
    label_scores = []
    for (n, seed, comp_idx), component in vals:
        comps.append(component.reshape(SHAPE))
        score = comp2score_when_active[n, seed, comp_idx]
        titles.append(f"{n} ({score:.3f})")
        label_scores.append(score)
    max_score_idx = np.argmax(label_scores)
    max_score = label_scores[max_score_idx]
    l2_maxscores[label] = (vals[max_score_idx][0], max_score, comps[max_score_idx])

    p50_score = np.median(label_scores)
    p75_score = np.percentile(label_scores, 75)

    cols = min(len(comps), 10)
    rows = math.ceil(len(comps) / cols)
    S(comps, (20,rows*3), cols, suptitle=f"{label} p50: {p50_score:.4f} p75: {p75_score:.4f}", ax_titles=titles)
    plt.show()

Ohk, we have the answers now. subjectively, i know what components are good.  

You basically look at their MSE differences. The clusters where the majorityt of the differences are very low, are not very useful. So we skip them.  
There is obviously some subjectivity here lols.   

Lets also keep the median scores in the suptitle

I kinda know which labels to take now. All we need to do now is to put those components in again, and retrain. Might decrease alpha a bit too maybe?  
Not sure.  Picking the max in each cluster of interest is fine.  


- pick the cluster labels you like
- pick the maximum score one in that, collect these components easy

After this is done, ill need to consolidate the code, and see if I can persist the clustering state (labels etc).  
This makes it easy to run in colab, save the state from time to time, and then check out stuff.  

For mass runs, ill also need to implement the model in JAX and vmap it for multiple seeds, this will do 10 seeds per comp, which is very fast.  
I would like to make it as un-attendable as possible.  

In [ ]:
labels_to_pick = [
    5, 10, 1, 4, 6, 0, 9
]

# find the max score component in each
# we want to check the overlap support matrix for them also

In [ ]:
S([l2_maxscores[l][-1].reshape(SHAPE) for l in labels_to_pick], (20,5), len(labels_to_pick))

In [ ]:
# now about overlap support
def support_overlap_matrix(W, threshold=0.01):
    # W: (n_components, n_dims)
    W = torch.tensor(W)
    abs_W = W.abs()
    maxvals = abs_W.max(dim=1, keepdim=True).values  # (n_components, 1)
    support = abs_W > threshold * maxvals  # (n_components, n_dims) binary

    # pairwise intersection over union (or just intersection)
    support_f = support.float()
    intersection = support_f @ support_f.T  # (n_components, n_components)
    support_sizes = support.sum(dim=1).float()  # (n_components,)
    min_sizes = torch.min(support_sizes.unsqueeze(1), support_sizes.unsqueeze(0))

    overlap = intersection / min_sizes  # normalized: 0=disjoint, 1=fully overlapping
    return overlap
W = np.array([l2_maxscores[l][-1].reshape(-1) for l in labels_to_pick])



In [ ]:
overlap_matrix = support_overlap_matrix(W)
overlap_matrix.fill_diagonal_(0)

In [ ]:
non_zero_pairs = np.argwhere(np.triu(overlap_matrix) != 0)

In [ ]:
np.unique(non_zero_pairs)

In [ ]:
for label in np.unique(non_zero_pairs):
    # for each label, find the other one
    overlappers = set()
    for p in non_zero_pairs:
        if p[0] == label:
            overlappers.add(p[1])
        elif p[1] == label:
            overlappers.add(p[0])
    print("label", label)
    S([W[label].reshape(SHAPE)] + [W[l].reshape(SHAPE) for l in overlappers], (20,3), len(overlappers)+1)
    plt.show()

good this is better.  
What can I do to make this more objective though now?   

For an overlapper, you have smaller components which are subset of it.  The sum of MSE of those components - the overlappers MSE is the point of contention.  
hmmmmmm. hmmmmmmmmmmmmmmmmmmmmm. hmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmm
How do i calculate the MSE now, without these methods? Can I actually rely on the algorithm to figure it out? Lets try this.  

In [ ]:
W.shape
S([w.reshape(SHAPE) for w in W], (20,3), len(W))

In [ ]:
np.save("./tmp-weights.npy", W)

In [ ]:
# for now, im removing the last one. I need a better way, but that can be thought of later.
# we do notice that the last one gets overwritten when we do the gradient descent again
# this is a promising direction, i might do it across different seeds to see which ones get deleted after checking the support overlap
# although, the method above which shows which thing overlaps with what is also a good subjective way of deciding which components to keep

In [ ]:
W_useful = W[:-1]

S([w.reshape(SHAPE) for w in W_useful], (20,3), len(W_useful))

In [ ]:
# scale using colab scaled weights. remember that this is important for us
# we would need a class which encapsulates the results of the whole run later
training_scaled_pw = torch.load("/Users/hariomnarang/Desktop/gdrive-sync/l2-o12-patches/cosine-anneal/scaled_pw.pt", weights_only=False)
# pw = patches_of_batch * weight
scaler = B.NormaliseStdScaler().fit(training_scaled_pw)
scaled_pw = scaler.transform(pw)


In [ ]:
del training_scaled_pw
gc.collect()

In [ ]:
# we have an initialised model now
# train the codes, freeze the weights and train the encoder only

model = TX.Autoencoder(scaled_pw.shape[1], W_useful.shape[0])
model.decoder.weight.data = torch.tensor(W_useful.T.copy(), dtype=torch.float32)
for param in model.decoder.parameters():
    param.requires_grad = False

run = TX.train(
    scaled_pw, 
    W.shape[0], 
    1e-2,
    device="mps", 
    epochs=800,
    baseline_epochs=600, 
    batch_size=64, 
    use_ln_term=False,
    recon_err_schedule=TX.CosineAnnealReconError(1000),
    initialised_model=model,
)

In [ ]:
run.loss

In [ ]:
run.components.shape

In [ ]:
torch.save(run.model.state_dict(), "./l2-o12-main-saved-model.pt")

In [ ]:
S([w.reshape(SHAPE) for w in W], (20,5), len(run.components))
plt.show()


In [ ]:
S([c.reshape(SHAPE) for c in run.components], (20,5), len(run.components))
plt.show()

In [ ]:
# interesting, it did remove the problematic child
gc.collect()

In [ ]:
def train_with_decoder_frozen(
    X, n_components, model, lr=1e-2, epochs=600, device="cpu", batch_size=256, verbose=True
):
    for param in model.decoder.parameters():
        param.requires_grad = False


    optimizer = torch.optim.Adam(model.encoder.parameters(), lr=lr)
    X_t = torch.tensor(X, dtype=torch.float32).to(device)
    model = model.to(device)
    for epoch in range(epochs):
        idx = torch.randperm(X_t.shape[0], device=device)
        permuted_X_t = X_t[idx]

        for i in range(0, permuted_X_t.shape[0], batch_size):
            batch = permuted_X_t[i:i+batch_size]

            recon, codes, _ = model(batch)
            loss = B.recon_loss(batch, recon, 1)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        if epoch % 50 == 0 and verbose:
            print(f"epoch {epoch} | recon_loss: {loss:.4f}")

    with torch.no_grad():
        recon, codes, _ = model(X_t)

    return B.SingleRun(
        model.to("cpu"),
        codes.to("cpu").numpy(),
        model.decoder.weight.T.detach().to("cpu").numpy(),
        recon.to("cpu").numpy(),
        ((X_t - recon) ** 2).to("cpu").mean(),
        {},
        None,
    )


In [ ]:
# nice, lets do a smaller intervention now
# first train by freezing the decoder, only train the encoder.  
model = TX.Autoencoder(scaled_pw.shape[1], W.shape[0])
model.decoder.weight.data = torch.tensor(W.T.copy(), dtype=torch.float32)

run_base = train_with_decoder_frozen(
    scaled_pw, W.shape[0], model, lr=1e-2, epochs=600, device="mps", batch_size=256, verbose=True
)

In [ ]:
model.decoder

In [ ]:
# we have trained the encoder only, now we do full training again, for lesser number of epochs though
# sigma_eps is expected as above (recon_loss)
# we simply sqrt the loss :)

In [ ]:
sigma_eps = np.sqrt(run.loss)
model = run.model
model.decoder.requires_grad_()
run = TX.train(
    scaled_pw, 
    W.shape[0], 
    1e-3,
    device="mps", 
    epochs=800, 
    batch_size=64, 
    use_ln_term=False,
    recon_err_schedule=TX.CosineAnnealReconError(1000),
    initialised_model=model,
    weights_algo="cyclic",
    sigma_eps_override=sigma_eps,
)

In [ ]:
S([w.reshape(SHAPE) for w in W], (20,5), len(run.components))
plt.show()

# interestingly, nothing changed here
# aah shit, i didnt unfreeze ;_;
S([c.reshape(SHAPE) for c in run.components], (20,5), len(run.components))
plt.show()

This does remove the main matrix and replace it with something else.  
Now what. This is inconclusive because my "assumption" of a good run is that the matrix would 0 out (it does change though, so thats kinda good news).  

It is also doing some shifting and all. hmmmmmm.  


This is inconclusive. fine. I might train it with some sparsity loss maybe to drive down the stuff which is not nice. But eh, nvm.  

The other thing we can do is simply train the overlapping components and see which change. that should be easy enough.  
Tis still not very nice though.   

Ohk im simply going to move forward with the components i have, the model's codes will simply be finetuned to these.   

In [ ]:
sigma_eps = np.sqrt(run.loss)
model = run.model
run = TX.train(
    scaled_pw, 
    W.shape[0], 
    1e-3,
    device="mps", 
    epochs=800, 
    batch_size=64, 
    use_ln_term=False,
    recon_err_schedule=TX.CosineAnnealReconError(1000),
    initialised_model=model,
    weights_algo="cyclic",
    sigma_eps_override=sigma_eps,
)

In [ ]:
# wtf the last one has a lot of MSE difference now whut
scores = get_comp_scores(scaled_pw, run.codes, run.components)
scores

In [ ]:
# it does not seem like the MSE difference is a very stable metric, wtf is happening here
# ive actually only fine tuned the codes of the main model
# oh wait, it might start doing overlaps like this hmmmmmm
# mmmmmmmmmmmm
scores = get_comp_scores(scaled_pw, run_base.codes, run_base.components)
scores

In [ ]:
S([w.reshape(SHAPE) for w in W], (20,5), len(run.components))
plt.show()

# interestingly, nothing changed here
# aah shit, i didnt unfreeze ;_;
S([c.reshape(SHAPE) for c in run_base.components], (20,5), len(run_base.components))
plt.show()

## aneal

In [ ]:
import pickle

dump_dir = MAIN_DUMP_DIR / "cosine-anneal"


In [ ]:
anneal_c2r = {}

for i in range(2, 16):
    d = dump_dir /f"r{i}.pkl"
    with open(d, "rb") as f:
        run = pickle.load(f)
        anneal_c2r[i] = run

In [ ]:
[anneal_c2r[i].loss for i in range(2,16)]

In [ ]:
plt.plot(list(range(2,16)), [anneal_c2r[i].loss for i in range(2,16)])
plt.show()

In [ ]:
import math

for i in range(2,16):
    cols = min(i, 10)
    rows = math.ceil(i / cols)
    rsize = 2*rows
    csize = 20
    scores, main_mse = get_comp_scores(scaled_pw, anneal_c2r[i].codes, anneal_c2r[i].components)
    scores= np.array(scores)
    # scores = (100*scores) / scores.sum()

    print("shape", i)
    S([c.reshape(SHAPE) for c in anneal_c2r[i].components], (csize, rsize), cols, viztype="local", suptitle=f"loss: {anneal_c2r[i].loss}", ax_titles=[f"{s:.3f}/{main_mse:.3f}" for s in scores])
    plt.show()

## across seeds

At seed 2, we see nice results. It gives essentially 2 sets of combinations.  
It seems the moment the first one is set to something, the other one is automatically derived.  


It might be useful to actually do this across more seeds then, since as the components increase, the combinations might too. But for now its fine.  


Surprisingly, the algorithm is very very stable yayyyy :)   


We start seeing splits in components=3. It might be useful to do an analysis of how much error each split has and how much each component contributes to the error.  
Looking at the MSE of each, we see that lower MSEs are winning.  


A problem with the current implementation is that it restricts a component to a strict place.  
What if we start out with all the components found in a seed, (increase n-components?). what happens then?   
The algo should hopefully settle on the set which are good. But there is the skew in our algo, which is a problem.  
We might want to do the initial batches without the skew.  

It seems like a good experiment lets see. I would like the other components to be empty though (only 3 components should have high.). That would be interesting.  
If in my loss i provide the number of components, can i add some constraint along the rows?  
Or can i add a new penalty? which might be just sparsifying or something?  

Hmmm. The best way rn is to first try it out though. Lets see.  

In [ ]:
seed_dump_dir = dump_dir / "comps"

In [ ]:
paths = [p for p in (seed_dump_dir / "3").glob("*.pkl")]
for p in paths:
    with open(p, "rb") as f:
        run = pickle.load(f)
    S([c.reshape(SHAPE) for c in run.components], (5,2), 3, suptitle=str(run.loss))
    plt.show()

## seed 3, all outputs

In [ ]:
from torch import nn
class Autoencoder(nn.Module):
    def __init__(self, input_dim, n_components):
        super().__init__()
        self.encoder = nn.Linear(input_dim, n_components, bias=False)
        self.decoder = nn.Linear(n_components, input_dim, bias=False)

    def forward(self, x):
        codes = self.encoder(x)
        latent = codes.unsqueeze(-1) * self.decoder.weight.T.unsqueeze(0)
        recon = latent.sum(dim=1)
        latent_perm = latent.permute(0, 2, 1)  # [batch, dims, n_components]
        return recon, codes, latent_perm

In [ ]:
import numpy as np
from dataclasses import dataclass
from typing import Any

@dataclass
class SingleRun:
    model: Any
    codes: np.ndarray
    components: np.ndarray
    recon: np.ndarray
    loss: float
    hyperparameters: dict
    baseline_loss: float | None = None

In [ ]:
seed12_paths =  Path.home() / "Desktop/gdrive-sync/l2-o12-patches/cosine-anneal/comps/12/pt"
runs12 = []
for pt in seed12_paths.glob("*.pt"):
    runs12.append(torch.load(pt, weights_only=False))


In [ ]:
! ls "$HOME/Desktop/gdrive-sync/l2-o12-patches/cosine-anneal/comps/10/"

In [ ]:
seed10_paths =  Path.home() / "Desktop/gdrive-sync/l2-o12-patches/cosine-anneal/comps/10/pt"
runs10 = []
for pt in seed10_paths.glob("*.pt"):
    runs10.append(torch.load(pt, weights_only=False))

In [ ]:
len(runs10)

In [ ]:
! ls {seed12_paths}/../../..

In [ ]:
scaled_pw = torch.load(seed12_paths.parent.parent.parent / "scaled_pw.pt", weights_only=False)

In [ ]:
for i in range(4):
    run = runs10[i]
    scores, main_mse = get_comp_scores(scaled_pw, run.codes, run.components)

    tot = sum(scores)
    idxs = torch.argsort(torch.tensor(scores), descending=True)
    S(
        [run.components[j].reshape(SHAPE) for j in idxs], 
        (20,2), 
        len(run.components), 
        suptitle=f"{i}/{main_mse}", 
        ax_titles=[f"{(100*scores[j]/tot):.4f}" for j in idxs]
    )
    plt.show()

In [ ]:
for i in range(4):
    run = runs12[i]
    scores, main_mse = get_comp_scores(scaled_pw, run.codes, run.components)

    tot = sum(scores)
    idxs = torch.argsort(torch.tensor(scores), descending=True)
    S(
        [run.components[j].reshape(SHAPE) for j in idxs], 
        (20,2), 
        len(run.components), 
        suptitle=f"{i}/{main_mse}", 
        ax_titles=[f"{(100*scores[j]/tot):.4f}" for j in idxs]
    )
    plt.show()

These are reasonably stable, for now im just picking the first one and continuing from 12 components.  
Now how do i know what these are doing?  

For an activation:
- extract all patches from the activation (from the code we used to initially extract the patches)
- get the pointwise sums
- run the model, get the coefficients.
- get the active components
- draw non-zero values of these components on the activation (or just side by side for now)


Later, Ill also need a simple classifier which says whether a patch is doable or not. For now, I'll simply use the older code which gets me patches with high attributions

## aneal and sgd with warm restarts

In [ ]:
import pickle

dump_dir = MAIN_DUMP_DIR / "cosine-anneal-and-sgd-with-warm-restarts"

anneal_c2r = {}

for i in range(2, 4):
    d = dump_dir /f"r{i}.pkl"
    with open(d, "rb") as f:
        run = pickle.load(f)
        anneal_c2r[i] = run

In [ ]:
import math

for i in range(2,4):
    cols = min(i, 10)
    rows = math.ceil(i / cols)
    rsize = 2*rows
    csize = 20
    scores, main_mse = get_comp_scores(scaled_pw, anneal_c2r[i].codes, anneal_c2r[i].components)
    scores= np.array(scores)
    # scores = (100*scores) / scores.sum()

    print("shape", i)
    S([c.reshape(SHAPE) for c in anneal_c2r[i].components], (csize, rsize), cols, viztype="local", suptitle=f"loss: {anneal_c2r[i].loss}", ax_titles=[f"{s:.3f}/{main_mse:.3f}" for s in scores])
    plt.show()

In [ ]:
plt.plot(list(range(2,4)), [])
plt.show()

## standard

In [ ]:
import pickle

dump_dir = MAIN_DUMP_DIR / "standard"

c2r = {}

for i in range(2, 16):
    d = dump_dir /f"r{i}.pkl"
    with open(d, "rb") as f:
        run = pickle.load(f)
        c2r[i] = run

In [ ]:
plt.plot(list(range(2,16)), [c2r[i].loss for i in range(2,16)])
plt.show()

In [ ]:
import math

for i in range(2,4):
    cols = min(i, 10)
    rows = math.ceil(i / cols)
    rsize = 2*rows
    csize = 20
    scores, main_mse = get_comp_scores(scaled_pw, c2r[i].codes, c2r[i].components)
    scores= np.array(scores)
    # scores = (100*scores) / scores.sum()

    print("shape", i)
    S([c.reshape(SHAPE) for c in c2r[i].components], (csize, rsize), cols, viztype="local", suptitle=f"loss: {c2r[i].loss}", ax_titles=[f"{s:.3f}/{main_mse:.3f}" for s in scores])
    plt.show()